In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
os.environ["OPENAI_API_KEY"] = "fake-key"

import csv
import json
import asyncio
from typing import List, Dict, Optional, Union

# 本地 ChatGLM-6B 推理
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("THUDM/chatglm-6b", trust_remote_code=True)
model = AutoModel.from_pretrained("THUDM/chatglm-6b", trust_remote_code=True).half().cuda()
model.eval()

# 项目依赖
from neuron_explainer.activations.activation_records import calculate_max_activation
from neuron_explainer.activations.activations import ActivationRecordSliceParams, load_neuron
from neuron_explainer.explanations.scoring import simulate_and_score
from neuron_explainer.explanations.explanations import SequenceSimulation
from neuron_explainer.activations.activation_records import (
    format_activation_records,
    non_zero_activation_proportion,
)
from neuron_explainer.explanations.few_shot_examples import ORIGINAL_EXAMPLES

def decode_tokens(tokens: List[str]) -> str:
    return "".join(tokens).replace("▁", " ").strip()

def simulate_and_score_sync(simulator, activation_records):
    """
    在同步环境里对异步函数 simulate_and_score(...) 进行调用。
    由于我们可能在 Jupyter 或已有事件循环的环境中，必须配合 nest_asyncio，否则会报错。
    """
    return asyncio.run(simulate_and_score(simulator, activation_records))

# ⚠️ 假设你已经正确设置 nest_asyncio + 模型加载等，略去重复部分，只展示优化点

class ChatGLM4Explainer:
    def __init__(self, api_key: str = None):
        pass

    def explain(self, activation_records) -> str:
        max_activation = calculate_max_activation(activation_records)

        system_prompt = (
            "We're studying neurons in a neural network. Each neuron looks for some particular thing in a short document. "
            "Look at the parts of the document the neuron activates for and summarize in a single sentence what the neuron is looking for. "
            "Don't list examples of words.\n\n"
            "The activation format is token<TAB>activation. Activation values range from 0 to 10. "
            "A neuron finding what it's looking for is represented by a non-zero activation value. "
            "The higher the activation value, the stronger the match."
            "Please do not list specific words."
        )

        few_shot_section = ""
        for i, example in enumerate(ORIGINAL_EXAMPLES[:2]):
            example_text = format_activation_records(
                example.activation_records,
                calculate_max_activation(example.activation_records),
                omit_zeros=False
            )
            filtered_text = ""
            if non_zero_activation_proportion(example.activation_records, calculate_max_activation(example.activation_records)) < 0.2:
                filtered_text = "\nSame activations, but with all zeros filtered out:\n" + \
                                format_activation_records(example.activation_records, calculate_max_activation(example.activation_records), omit_zeros=True)
            few_shot_section += (
                f"Neuron {i+1}\n"
                f"Activations:\n{example_text}"
                f"{filtered_text}\n\n"
                f"Explanation of neuron {i+1} behavior: the main thing this neuron does is find {example.explanation}.\n\n"
            )

        current_text = format_activation_records(activation_records, max_activation, omit_zeros=False)
        filtered_text = ""
        if non_zero_activation_proportion(activation_records, max_activation) < 0.2:
            filtered_text = "\nSame activations, but with all zeros filtered out:\n" + \
                            format_activation_records(activation_records, max_activation, omit_zeros=True)

        current_section = (
            f"Neuron 3\n"
            f"Activations:\n{current_text}"
            f"{filtered_text}\n\n"
            f"Explanation of neuron 3 behavior: the main thing this neuron does is find\n"
        )

        prompt = system_prompt + "\n\n" + few_shot_section + current_section
        response, _ = model.chat(tokenizer, prompt, history=[])

        explanation = response.strip()
        if not explanation or len(explanation.split()) < 3:
            print("[!] Warning: Explanation seems too short or empty.")
        #print("[Explanation]", explanation)
        return explanation


class ChatGLMNeuronSimulator:
    def __init__(self, api_key: str, explanation: str):
        self.explanation = explanation

    def simulate_activations(self, tokens: List[str]) -> Dict[str, float]:
        def format_record(tokens: List[str], activations: Optional[List[Union[int, str]]] = None) -> str:
            return "\n".join(
                f"{token}\t{activations[i] if activations else 'unknown'}" for i, token in enumerate(tokens)
            )

        few_shot = (
            "Neuron 1\n"
            "Explanation: words related to community\n"
            "Tokens:\n"
            "the\tunknown\n"
            "sense\tunknown\n"
            "of\tunknown\n"
            "together\tunknown\n"
            "ness\tunknown\n"
            "\nNeuron 1 Activations:\n"
            "the\t0\n"
            "sense\t0\n"
            "of\t0\n"
            "together\t4\n"
            "ness\t7\n"
        )

        formatted_tokens = format_record(tokens)
        user_prompt = (
            f"{few_shot}\n\n"
            f"Neuron 2\nExplanation: {self.explanation.strip()}\n"
            f"Tokens:\n{formatted_tokens}\n\n"
            f"Neuron 2 Activations:"
        )

        full_prompt = (
            "We're studying neurons in a neural network.\n"
            "Each neuron looks for some particular thing in a short document.\n"
            "You will be given an explanation of the neuron's behavior and a list of tokens.\n"
            "Your task is to output predicted activations for each token.\n\n"
            "**Output Format Requirement:**\n"
            "- ONLY output lines in the format: token<TAB>activation\n"
            "- Each activation should be a float number between 0 and 10\n"
            "- DO NOT include any explanation or commentary\n"
            "- DO NOT skip any token; output must include an activation for every token provided\n"
            "- Example: \n"
            "the\t0\nsense\t0\nof\t0\ntogether\t4\nness\t7\n\n"
            + user_prompt
        )

        response, _ = model.chat(tokenizer, full_prompt, history=[])
        lines = response.strip().splitlines()
        activation_map = {}

        for line in lines:
            if "\t" in line:
                try:
                    token, act = line.strip().split("\t")
                    activation_map[token] = float(act)
                except:
                    activation_map[token] = 0.0

        #print("[Simulator Output]", activation_map)

        #if all(v == 0.0 for v in activation_map.values()):
            #print("[!] Warning: All simulated activations are 0.")

        return {token: activation_map.get(token, 0.0) for token in tokens}


# 评分模块包装
class ChatGLMNeuronSimulatorForScoring:
    def __init__(self, glm_simulator: ChatGLMNeuronSimulator):
        self.glm_simulator = glm_simulator

    async def simulate(self, tokens: List[str]) -> SequenceSimulation:
        sim_dict = self.glm_simulator.simulate_activations(tokens)
        predicted_activations = [sim_dict.get(tok, 0.0) for tok in tokens]
        #if all(v == 0.0 for v in predicted_activations):
            #print("[!] Warning: All predicted_activations are 0 in SequenceSimulation")
        return SequenceSimulation(
            tokens=tokens,
            expected_activations=predicted_activations,
            activation_scale=10.0,
            distribution_values=[],
            distribution_probabilities=[],
        )


# -------------------------
# 主逻辑优化
# -------------------------
csv_file = "chatglm6b_local_scores_random.csv"
fieldnames = ["Neuron", "Explanation", "Score", "Error"]
import random

with open(csv_file, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    
    random_neuron_indices = random.sample(range(0, 4096), 400)
    
    for neuron_index in random_neuron_indices:
        try:
            neuron_record = load_neuron(27, neuron_index)
            slice_params = ActivationRecordSliceParams(n_examples_per_split=3)
            train_records = neuron_record.train_activation_records(slice_params)
            valid_records = neuron_record.valid_activation_records(slice_params)

            explainer = ChatGLM4Explainer()
            explanation = explainer.explain(train_records)

            glm_sim = ChatGLMNeuronSimulator(api_key=None, explanation=explanation)
            wrapped_sim = ChatGLMNeuronSimulatorForScoring(glm_sim)

            scored = simulate_and_score_sync(wrapped_sim, valid_records)
            score = scored.get_preferred_score()

            writer.writerow({
                "Neuron": neuron_index,
                "Explanation": explanation,
                "Score": f"{score:.4f}",
                "Error": ""
            })
            print(f"[✓] Neuron {neuron_index}: Score {score:.4f}")

        except Exception as e:
            writer.writerow({
                "Neuron": neuron_index,
                "Explanation": "",
                "Score": "",
                "Error": str(e)
            })
            print(f"[!] Neuron {neuron_index} failed: {e}")